# Comment 7 — Resolution Ablation: Native vs 256×256 on MoNuSAC

## Experimental design

The trained 256×256 model is held **fixed** throughout. Only the input
resolution changes:

| Condition | Input to model | GT mask resolution | CAM output |
|---|---|---|---|
| **Resized (baseline)** | 256×256 | 256×256 | 256×256 |
| **Native** | original H×W | original H×W | original H×W |

The model was trained on 256×256 patches, so the native-resolution
inference is deliberately out-of-distribution with respect to spatial
scale. DenseNet169 is fully convolutional up to the global average
pooling layer, so it accepts any input size ≥ 32×32 without modification.
The CAMEngine reads `H, W` from the input tensor and upsamples heatmaps
to match, so no engine changes are needed.

**What this ablation answers:**  
The main paper found that the FPN pyramid main effect is *reversed* on
MoNuSAC (−0.0069 vs +0.0092 on PanNuke). The proposed mechanism is that
downsampling variable-size native patches to 256×256 destroys
nucleus-scale spatial structure in the shallow feature maps that the FPN
early layers rely on. If this is correct, running CAMs on native-resolution
images should:
1. Recover positive FPN pyramid main effect (or reduce the reversal)
2. Widen the gap between FPN and single-layer methods
3. Show the largest improvement on images with large native resolution
   (where 256×256 resizing caused the most spatial compression)

**Prerequisites:**  
`pannuke_monusac_cam.ipynb` must have been run first to produce
`outputs/monusac_cam_comparison/best_densenet169_monusac.pth` and
`outputs/monusac_cam_comparison/results_50_test.csv` (the 256×256 baseline).

## 1. Imports & Environment

In [ ]:
import gc
import json
import random
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 2. Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
MONUSAC_ROOT  = Path('MoNuSAC_images_and_annotations')
OUTPUT_ROOT   = Path('./outputs')
GRAD_OUT_DIR  = OUTPUT_ROOT / 'monusac_cam_comparison'
MODEL_PATH    = GRAD_OUT_DIR / 'best_densenet169_monusac.pth'
BASELINE_CSV  = GRAD_OUT_DIR / 'results_50_test.csv'   # 256×256 results

OUT_DIR  = OUTPUT_ROOT / 'monusac_resolution_ablation'
FIG_DIR  = OUT_DIR / 'figures'
TRIP_DIR = OUT_DIR / 'triplets'
for d in [OUT_DIR, FIG_DIR, TRIP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Class definitions (identical to main notebook) ────────────────────────────
CLASS_NAMES:   List[str] = ['Epithelial', 'Lymphocyte', 'Macrophage', 'Neutrophil']
CLASS_DISPLAY: List[str] = ['Epithelial', 'Lymphocyte', 'Macrophage', 'Neutrophil']
NUM_CLASSES = 4

XML_CLASS_MAP: Dict[str, int] = {
    'Epithelial': 0, 'epithelial': 0, 'EPITHELIAL': 0,
    'Lymphocyte': 1, 'lymphocyte': 1, 'LYMPHOCYTE': 1,
    'Macrophage': 2, 'macrophage': 2, 'MACROPHAGE': 2,
    'Neutrophil': 3, 'neutrophil': 3, 'NEUTROPHIL': 3,
    'Ambiguous': -1, 'ambiguous': -1,
}

OFFICIAL_TEST_PATIENTS: List[str] = [
    'TCGA-55-1594-01Z-00-DX1', 'TCGA-67-3771-01A-01-BS1',
    'TCGA-69-7760-01A-01-BS1', 'TCGA-73-4668-01A-01-BS1',
    'TCGA-EJ-5510-01A-01-BS1', 'TCGA-G9-6336-01A-01-BS1',
    'TCGA-G9-6348-01A-01-BS1', 'TCGA-VP-A878-01A-01-BS1',
    'TCGA-A2-A0CV-01A-02-TSB', 'TCGA-A7-A0CE-01A-11-TS2',
    'TCGA-D8-A1JF-01A-01-TS1', 'TCGA-E2-A14N-01A-02-TSB',
    'TCGA-2Z-A9J9-01A-01-TS1', 'TCGA-B0-5698-01A-01-BS1',
    'TCGA-B0-5710-01A-01-BS1', 'TCGA-B0-5711-01A-01-BS1',
]

# ── Shared constants ──────────────────────────────────────────────────────────
IMG_SIZE     = 256      # model was trained at this resolution
THRESHOLD    = 0.5
IOU_THRS     = [0.25, 0.40, 0.50]
NOISE_THR    = 0.25
ALPHA_SCALE  = 0.85
DROPOUT_P    = 0.3
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Method registry (4 gradient methods; 2 conditions each = 8 result columns) ─
METHODS = ['std_gradcam', 'fpn_gradcam', 'std_layercam', 'fpn_layercam']
METHOD_LABELS = {
    'std_gradcam'  : 'Standard GradCAM',
    'fpn_gradcam'  : 'FPN-GradCAM',
    'std_layercam' : 'Standard LayerCAM',
    'fpn_layercam' : 'FPN-LayerCAM',
}
METHOD_COLORS = {
    'std_gradcam'  : '#90CAF9',
    'fpn_gradcam'  : '#EF9A9A',
    'std_layercam' : '#A5D6A7',
    'fpn_layercam' : '#FFE082',
}
METHOD_MARKERS = {
    'std_gradcam': 'o', 'fpn_gradcam': 's',
    'std_layercam': '^', 'fpn_layercam': 'D',
}
CLASS_COLORS = ['#E53935', '#1E88E5', '#FFB300', '#43A047']
CLASS_COLORS_F32 = [
    (int(h[1:3],16)/255, int(h[3:5],16)/255, int(h[5:7],16)/255)
    for h in CLASS_COLORS
]

FPN_LAYERS: List[Tuple[str, float]] = [
    ('backbone.features.denseblock1', 0.20),
    ('backbone.features.denseblock2', 0.35),
    ('backbone.features.denseblock4', 0.45),
]

print('Configuration loaded.')
print(f'Baseline (256×256) CSV : {BASELINE_CSV}')
print(f'Native-resolution output: {OUT_DIR}')

## 3. XML Parser and Index Builder

In [ ]:
def parse_monusac_xml(xml_path: Path, img_h: int, img_w: int) -> np.ndarray:
    """Parse MoNuSAC XML → (H, W, 4) uint8 mask at original resolution."""
    mask = np.zeros((img_h, img_w, NUM_CLASSES), dtype=np.uint8)
    try:
        root = ET.parse(str(xml_path)).getroot()
    except ET.ParseError:
        return mask
    for ann in root.findall('Annotation'):
        class_idx = -1
        attrs = ann.find('Attributes')
        if attrs is not None:
            first = attrs.find('Attribute')
            if first is not None:
                raw = first.get('Name', '').strip()
                class_idx = XML_CLASS_MAP.get(raw,
                            XML_CLASS_MAP.get(raw.capitalize(), -1))
        if class_idx < 0:
            raw_name  = ann.get('Name', ann.get('PartOfGroup', ''))
            class_idx = XML_CLASS_MAP.get(raw_name.strip(), -1)
        if class_idx < 0:
            continue
        for region in ann.findall('.//Region'):
            vertices = region.findall('.//Vertex')
            if len(vertices) < 3:
                continue
            pts = []
            for v in vertices:
                x = v.get('X') or v.get('x')
                y = v.get('Y') or v.get('y')
                if x and y:
                    pts.append([float(x), float(y)])
            if len(pts) < 3:
                continue
            pts_arr = np.array(pts, dtype=np.float32)
            pts_arr[:, 0] = np.clip(pts_arr[:, 0], 0, img_w - 1)
            pts_arr[:, 1] = np.clip(pts_arr[:, 1], 0, img_h - 1)
            pts_int = pts_arr.astype(np.int32).reshape((-1, 1, 2))
            ch = np.ascontiguousarray(mask[..., class_idx])
            cv2.fillPoly(ch, [pts_int], color=1)
            mask[..., class_idx] = ch
    return mask


def build_monusac_index(root: Path) -> pd.DataFrame:
    """Scan MONUSAC_ROOT. Adds orig_h, orig_w columns for each image."""
    records = []
    for pdir in sorted(d for d in root.iterdir() if d.is_dir()):
        pid     = pdir.name
        is_test = pid in OFFICIAL_TEST_PATIENTS
        for tif in sorted(pdir.glob('*.tif')):
            xml = tif.with_suffix('.xml')
            if not xml.exists():
                continue
            # Read native dimensions without loading full image data
            with Image.open(tif) as im:
                orig_w, orig_h = im.size
            records.append({
                'patient_id' : pid,
                'sample_name': tif.stem,
                'tif_path'   : str(tif),
                'xml_path'   : str(xml),
                'is_test'    : is_test,
                'orig_h'     : orig_h,
                'orig_w'     : orig_w,
            })
    df = pd.DataFrame(records)
    print(f'Index: {len(df)} samples  '
          f'(test={df.is_test.sum()}, train={( ~df.is_test).sum()})')
    print(f'Native size range:  '
          f'H {df.orig_h.min()}–{df.orig_h.max()}  '
          f'W {df.orig_w.min()}–{df.orig_w.max()}')
    return df


print('Scanning MoNuSAC directory...')
df_index = build_monusac_index(MONUSAC_ROOT)

# Force same random 80/20 split as main notebook (seed=42)
rng_split    = np.random.default_rng(SEED)
all_patients = df_index['patient_id'].unique().copy()
rng_split.shuffle(all_patients)
n_test_pts   = max(1, int(len(all_patients) * 0.2))
test_pts     = set(all_patients[:n_test_pts])
df_index['is_test'] = df_index['patient_id'].isin(test_pts)

test_global_indices = np.where(df_index['is_test'].values)[0]
test_cam_indices    = np.sort(test_global_indices.copy())
N_TEST = len(test_cam_indices)
print(f'\nForced random split (seed={SEED}):')
print(f'  Train/val patients : {len(all_patients)-n_test_pts}')
print(f'  Test patients      : {n_test_pts}')
print(f'  Test images        : {N_TEST}')

## 4. Model

In [ ]:
class DenseNet169MultiLabel(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, dropout_p: float = 0.3):
        super().__init__()
        backbone = models.densenet169(weights=None)
        in_features = backbone.classifier.in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.backbone(x))


model = DenseNet169MultiLabel(num_classes=NUM_CLASSES, dropout_p=DROPOUT_P).to(DEVICE)
ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded checkpoint — epoch {ckpt["epoch"]}, val AUC {ckpt["val_auc"]:.4f}')
print('Model trained at 256×256 — will be queried at native resolution in this notebook.')

## 5. Native-Resolution Preprocessing & Inference Utilities

The key difference from the main notebook: the transform **does not resize**.
The image is only converted to tensor and normalised with ImageNet statistics.
The model's fully-convolutional backbone accepts any spatial size ≥ 32×32;
only the final `Linear(1664→512)` layer requires a fixed feature vector,
which DenseNet's global average pooling provides regardless of spatial size.

In [ ]:
# ── 256×256 transform (for the resized baseline condition) ────────────────────
_tfm_256 = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ── Native-resolution transform (no spatial resize) ───────────────────────────
# Applied to the original image dimensions straight from the .tif file.
# ToTensor() handles arbitrary HxW; Normalize() is spatially invariant.
_tfm_native = T.Compose([
    T.ToTensor(),                         # HWC uint8 → CHW float32 [0,1]
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def preprocess_native(img_rgb: np.ndarray) -> torch.Tensor:
    """HWC uint8 → (1,3,H,W) float32 at NATIVE resolution, on DEVICE.
    requires_grad=True so backward() works for gradient-based CAMs.
    """
    return (_tfm_native(img_rgb)
            .unsqueeze(0).to(DEVICE).requires_grad_(True))


def preprocess_256(img_rgb: np.ndarray) -> torch.Tensor:
    """HWC uint8 → (1,3,256,256) float32 resized, on DEVICE."""
    return (_tfm_256(img_rgb)
            .unsqueeze(0).to(DEVICE).requires_grad_(True))


@torch.no_grad()
def get_pred(tensor: torch.Tensor) -> Tuple[np.ndarray, np.ndarray]:
    """Returns (binary_labels, sigmoid_probs) for any spatial size input."""
    logits = model(tensor.detach())
    probs  = torch.sigmoid(logits).squeeze().cpu().numpy()
    return (probs >= THRESHOLD).astype(np.uint8), probs


def compute_iou(cam: np.ndarray, gt_ch: np.ndarray, thr: float) -> float:
    gt_bin = gt_ch > 0
    if not gt_bin.any():
        return float('nan')
    pred  = cam >= thr
    inter = (pred & gt_bin).sum()
    union = (pred | gt_bin).sum()
    return float(inter) / float(union) if union > 0 else float('nan')


def resize_ch(ch: np.ndarray, hw: Tuple[int, int]) -> np.ndarray:
    """Resize a single mask channel to (H, W) using nearest-neighbour."""
    if ch.shape == hw:
        return ch
    return cv2.resize(ch.astype(np.float32),
                      (hw[1], hw[0]),
                      interpolation=cv2.INTER_NEAREST)


def render_gt_mask(mask_4ch: np.ndarray,
                   hw: Tuple[int, int]) -> np.ndarray:
    H, W   = hw
    canvas = np.zeros((H, W, 3), dtype=np.float32)
    oh, ow = mask_4ch.shape[:2]
    if (oh, ow) != (H, W):
        mask_4ch = np.stack([
            cv2.resize(mask_4ch[..., c], (W, H),
                       interpolation=cv2.INTER_NEAREST)
            for c in range(NUM_CLASSES)
        ], axis=-1)
    for c_idx, color in enumerate(CLASS_COLORS_F32):
        canvas[mask_4ch[..., c_idx] > 0] = color
    return (canvas * 255).clip(0, 255).astype(np.uint8)


print('Preprocessing utilities defined.')
print('  _tfm_256    : Resize(256) → ToTensor → Normalize')
print('  _tfm_native : ToTensor → Normalize  (no resize)')

## 6. CAMEngine

Identical to `pannuke_monusac_cam.ipynb`. The engine reads `H, W` from
`tensor.shape[2:4]` so it automatically produces heatmaps at whatever
resolution the input tensor has — no modification needed.

In [ ]:
class CAMEngine:
    def __init__(self, model: nn.Module,
                 use_pyramid: bool  = False,
                 use_layercam: bool = False,
                 sharpen: bool      = True) -> None:
        self.model        = model
        self.use_pyramid  = use_pyramid
        self.use_layercam = use_layercam
        self.sharpen      = sharpen and use_pyramid
        self._handles: List = []
        self._layers: List[Tuple[str, float]] = (
            FPN_LAYERS if use_pyramid
            else [('backbone.features.denseblock4', 1.0)]
        )
        self._store: Dict[str, Dict] = {
            n: {'act': None, 'grad': None} for n, _ in self._layers
        }
        self._register_hooks()

    def _sub(self, dotpath: str) -> nn.Module:
        m = self.model
        for k in dotpath.split('.'): m = getattr(m, k)
        return m

    def _register_hooks(self) -> None:
        for name, _ in self._layers:
            mod = self._sub(name); n = name
            self._handles.append(mod.register_forward_hook(
                lambda m, i, o, n=n: self._store[n].__setitem__('act', o.detach())))
            self._handles.append(mod.register_full_backward_hook(
                lambda m, gi, go, n=n: self._store[n].__setitem__('grad', go[0].detach())))

    def remove_hooks(self) -> None:
        for h in self._handles: h.remove()
        self._handles.clear()

    def _compute_level(self, act: torch.Tensor,
                       grad: torch.Tensor,
                       hw: Tuple[int, int]) -> np.ndarray:
        if self.use_layercam:
            cam = F.relu((F.relu(grad) * act).sum(dim=1, keepdim=True))
        else:
            alpha = grad.mean(dim=(2, 3), keepdim=True)
            cam   = F.relu((alpha * act).sum(dim=1, keepdim=True))
        arr    = cam.squeeze().cpu().numpy()
        arr    = np.maximum(arr, 0)
        interp = cv2.INTER_CUBIC if self.use_pyramid else cv2.INTER_LINEAR
        arr    = cv2.resize(arr, (hw[1], hw[0]), interpolation=interp)
        if arr.max() > 1e-8: arr /= arr.max()
        return arr.astype(np.float32)

    @staticmethod
    def _bilateral_sharpen(cam: np.ndarray,
                           guide: np.ndarray) -> np.ndarray:
        gray = cv2.cvtColor(guide, cv2.COLOR_RGB2GRAY)
        u8   = (cam * 255).astype(np.uint8)
        if hasattr(cv2, 'ximgproc') and hasattr(cv2.ximgproc, 'jointBilateralFilter'):
            s = cv2.ximgproc.jointBilateralFilter(gray, u8, 9, 75, 75)
        else:
            s = cv2.bilateralFilter(u8, 9, 75, 75)
        r = s.astype(np.float32) / 255.0
        if r.max() > 1e-8: r /= r.max()
        return r

    def generate(self, tensor: torch.Tensor,
                 class_idx: int,
                 guide: Optional[np.ndarray] = None) -> np.ndarray:
        # H, W are read from the tensor — works at any resolution
        H, W = tensor.shape[2], tensor.shape[3]
        self.model.zero_grad()
        self.model(tensor)[0, class_idx].backward(retain_graph=False)
        cams, weights = [], []
        for name, w in self._layers:
            a = self._store[name]['act']
            g = self._store[name]['grad']
            if a is not None and g is not None:
                cams.append(self._compute_level(a, g, (H, W)))
                weights.append(w)
        if not cams:
            return np.zeros((H, W), dtype=np.float32)
        total = sum(weights)
        fused = sum(w / total * c for w, c in zip(weights, cams))
        if fused.max() > 1e-8: fused /= fused.max()
        if self.sharpen and guide is not None:
            guide_r = cv2.resize(guide, (W, H))
            fused   = self._bilateral_sharpen(fused, guide_r)
        return fused.astype(np.float32)


def make_engines(mdl: nn.Module) -> Dict[str, CAMEngine]:
    return {
        'std_gradcam'  : CAMEngine(mdl, use_pyramid=False, use_layercam=False),
        'fpn_gradcam'  : CAMEngine(mdl, use_pyramid=True,  use_layercam=False),
        'std_layercam' : CAMEngine(mdl, use_pyramid=False, use_layercam=True),
        'fpn_layercam' : CAMEngine(mdl, use_pyramid=True,  use_layercam=True),
    }


engines_native = make_engines(model)
print('CAMEngines instantiated (native-resolution condition).')
print('generate() reads H,W from input tensor — no code change needed for native res.')

## 7. Native-Resolution CAM Loop

For each test image:
1. Load the original `.tif` at **native resolution** (variable H×W)
2. Parse the XML mask at native resolution
3. Preprocess with `_tfm_native` (no resize)
4. Generate all 4 CAMs — the engine upsamples to native H×W automatically
5. Evaluate IoU between the native-resolution CAM and the native-resolution GT mask
6. Save a triplet figure at native resolution for qualitative comparison

> **Memory note (8 GB VRAM):** Native images range from ~350×350 to ~1000×1000 px.
> A 1000×1000 input tensor is ~12 MB; the backward pass for one class takes
> ~4× that. Peak usage per image is <200 MB — well within budget.
> `del t; torch.cuda.empty_cache()` runs after each image to prevent
> fragmentation across variable-size allocations.

In [ ]:
native_rows: List[Dict] = []
(TRIP_DIR / 'native').mkdir(exist_ok=True)

print(f'Running 4 CAM methods at NATIVE resolution on {N_TEST} test images...')

for local_i, gidx in enumerate(tqdm(test_cam_indices, desc='Native CAM')):
    row_info = df_index.iloc[gidx]
    tif_path = Path(row_info['tif_path'])
    xml_path = Path(row_info['xml_path'])

    # ── Load image at native resolution ──────────────────────────────────────
    img_pil  = Image.open(tif_path).convert('RGB')
    orig_w, orig_h = img_pil.size             # PIL: (w, h)
    img_native = np.array(img_pil, dtype=np.uint8)   # (orig_h, orig_w, 3)

    # ── Parse GT mask at native resolution ───────────────────────────────────
    mask_native = parse_monusac_xml(
        xml_path, orig_h, orig_w
    ).astype(np.float32)                      # (orig_h, orig_w, 4)

    # ── Multi-hot labels from native mask ────────────────────────────────────
    gt_labels = (mask_native.sum(axis=(0, 1)) > 0).astype(np.uint8)  # (4,)

    # ── Prediction at native resolution ──────────────────────────────────────
    t = preprocess_native(img_native)         # (1, 3, orig_h, orig_w)
    pred_lbl, probs = get_pred(t)
    active = [i for i, p in enumerate(pred_lbl) if p == 1]

    row: Dict = {
        'image_idx'  : local_i,
        'global_idx' : int(gidx),
        'patient_id' : row_info['patient_id'],
        'sample_name': row_info['sample_name'],
        'orig_h'     : orig_h,
        'orig_w'     : orig_w,
        'cardinality': int(gt_labels.sum()),
        'scale_factor': round(max(orig_h, orig_w) / IMG_SIZE, 3),
    }
    for ci, cname in enumerate(CLASS_NAMES):
        row[f'gt_{cname}']   = int(gt_labels[ci])
        row[f'pred_{cname}'] = int(pred_lbl[ci])
        row[f'prob_{cname}'] = round(float(probs[ci]), 4)

    # ── Generate 4 CAMs at native resolution ─────────────────────────────────
    all_cams: Dict[str, Dict[int, np.ndarray]] = {m: {} for m in METHODS}
    for c_idx in active:
        for mname, engine in engines_native.items():
            t_cam = preprocess_native(img_native)
            cam   = engine.generate(t_cam, c_idx, guide=img_native)
            all_cams[mname][c_idx] = cam   # shape: (orig_h, orig_w)
            del t_cam
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    # ── IoU at native resolution (CAM vs native GT mask) ─────────────────────
    for ci, cname in enumerate(CLASS_NAMES):
        gt_ch = mask_native[..., ci]          # already (orig_h, orig_w)
        for mname in METHODS:
            cam = all_cams[mname].get(
                ci, np.zeros((orig_h, orig_w), dtype=np.float32)
            )
            for thr in IOU_THRS:
                row[f'{mname}_iou_{thr:.2f}_{cname}'] = round(
                    compute_iou(cam, gt_ch, thr), 4)
            row[f'{mname}_area_{cname}'] = round(
                float((cam >= NOISE_THR).mean()), 4)

    native_rows.append(row)

    # ── Triplet figure (first active class, native resolution) ────────────────
    if active and local_i < 20:   # save first 20 to keep disk usage reasonable
        c_show     = active[0]
        cname_show = CLASS_NAMES[c_show]
        cam_show   = all_cams['fpn_layercam'].get(
            c_show, np.zeros((orig_h, orig_w), dtype=np.float32)
        )
        gt_vis  = render_gt_mask(mask_native, (orig_h, orig_w))
        r, g2, b2 = CLASS_COLORS_F32[c_show]
        clean     = np.where(cam_show >= NOISE_THR, cam_show, 0.0)
        comp_c    = np.stack([clean*r, clean*g2, clean*b2], axis=-1)
        alpha     = (clean * ALPHA_SCALE)[..., None]
        bg        = img_native.astype(np.float32) / 255.0
        overlay   = ((comp_c * alpha + bg * (1-alpha)).clip(0,1) * 255).astype(np.uint8)
        iou_v     = row.get(f'fpn_layercam_iou_0.50_{cname_show}', float('nan'))

        fig = plt.figure(figsize=(18, 5.6), facecolor='#0d0d0d')
        gs  = fig.add_gridspec(1, 3, wspace=0.04)
        for col_i, (panel, title) in enumerate([
            (img_native, f'Original ({orig_h}×{orig_w})'),
            (gt_vis,     'GT Mask (native res)'),
            (overlay,    'FPN-LayerCAM (native res)'),
        ]):
            ax = fig.add_subplot(gs[0, col_i])
            ax.imshow(panel); ax.axis('off')
            ax.set_title(title, color='white', fontsize=9, fontweight='bold', pad=4)
        iou_s = f'{iou_v:.3f}' if iou_v == iou_v else 'N/A'
        fig.axes[2].legend(
            handles=[mpatches.Patch(
                facecolor=CLASS_COLORS_F32[c_show], edgecolor='white', lw=0.4,
                label=f'{cname_show}  IoU@0.50={iou_s}  '
                      f'scale={row["scale_factor"]:.2f}×'
            )],
            loc='lower right', fontsize=7.5, framealpha=0.8,
            facecolor='#111111', edgecolor='#555555', labelcolor='white',
        )
        fig.suptitle(
            f'#{local_i:02d}  |  {row_info["sample_name"][:30]}  |  '
            f'Native {orig_h}×{orig_w}  (model trained at 256×256)',
            color='#dddddd', fontsize=9, fontweight='bold', y=1.02,
        )
        plt.savefig(TRIP_DIR / 'native' / f'native_{local_i:03d}.png',
                    dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
        plt.close(fig)

    del t, all_cams
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

for eng in engines_native.values():
    eng.remove_hooks()

df_native = pd.DataFrame(native_rows)
# Fill missing IoU columns with NaN
for mname in METHODS:
    for cname in CLASS_NAMES:
        for thr in IOU_THRS:
            col = f'{mname}_iou_{thr:.2f}_{cname}'
            if col not in df_native.columns:
                df_native[col] = np.nan

df_native.to_csv(OUT_DIR / 'results_native_test.csv', index=False)
print(f'\nNative results saved. Shape: {df_native.shape}')
print(f'Native size distribution:')
print(df_native[['orig_h','orig_w','scale_factor']].describe().round(1))

## 8. Load 256×256 Baseline Results

In [ ]:
# Try both filename variants
baseline_candidates = [
    GRAD_OUT_DIR / 'results_50_test.csv',
    GRAD_OUT_DIR / 'results_test.csv',
]
baseline_csv = next((p for p in baseline_candidates if p.exists()), None)
if baseline_csv is None:
    raise FileNotFoundError(
        'Baseline 256×256 results CSV not found. '
        'Run pannuke_monusac_cam.ipynb first.'
    )

df_256 = pd.read_csv(baseline_csv)
print(f'Baseline (256×256) loaded : {df_256.shape}  from {baseline_csv.name}')
print(f'Native             loaded : {df_native.shape}')

# Align on global_idx — both DataFrames should cover the same test images
common_gidx = set(df_256['global_idx']) & set(df_native['global_idx'])
df_256_a    = df_256[df_256['global_idx'].isin(common_gidx)].sort_values('global_idx').reset_index(drop=True)
df_nat_a    = df_native[df_native['global_idx'].isin(common_gidx)].sort_values('global_idx').reset_index(drop=True)
print(f'Aligned on global_idx: {len(common_gidx)} images')

## 9. IoU Summary: Native vs 256×256

In [ ]:
def macro_iou(df: pd.DataFrame, mkey: str, thr: float = 0.50) -> float:
    thr_s = f'{thr:.2f}'
    vals  = [df[f'{mkey}_iou_{thr_s}_{c}'].dropna().mean()
             for c in CLASS_NAMES
             if f'{mkey}_iou_{thr_s}_{c}' in df.columns]
    return float(np.nanmean(vals)) if vals else float('nan')


def per_class_iou(df: pd.DataFrame, mkey: str, thr: float = 0.50) -> Dict[str, float]:
    thr_s = f'{thr:.2f}'
    return {c: round(df[f'{mkey}_iou_{thr_s}_{c}'].dropna().mean(), 4)
            for c in CLASS_NAMES
            if f'{mkey}_iou_{thr_s}_{c}' in df.columns}


summary_rows = []
for mkey in METHODS:
    for cond, df_c in [('256x256', df_256_a), ('native', df_nat_a)]:
        m256 = macro_iou(df_c, mkey)
        pc   = per_class_iou(df_c, mkey)
        summary_rows.append({
            'method'    : METHOD_LABELS[mkey],
            'condition' : cond,
            'macro_iou' : round(m256, 4),
            **{f'iou_{c}': pc.get(c, float('nan')) for c in CLASS_NAMES},
        })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(OUT_DIR / 'resolution_ablation_summary.csv', index=False)

print(f'IoU@0.50 — Native vs 256×256  (n={len(common_gidx)} test images)\n')
print(f'{"Method":<28}  {"256×256":>9}  {"Native":>9}  {"Δ(nat-256)":>12}')
print('-' * 65)
for mkey in METHODS:
    v256 = macro_iou(df_256_a, mkey)
    vnat = macro_iou(df_nat_a, mkey)
    print(f'{METHOD_LABELS[mkey]:<28}  {v256:>9.4f}  {vnat:>9.4f}  {vnat-v256:>+12.4f}')

print(f'\nPer-class detail (IoU@0.50):')
print(f'{"Method":<28}  {"Cond":>7}', end='')
for c in CLASS_NAMES:
    print(f'  {c[:10]:>10}', end='')
print()
print('-' * 85)
for mkey in METHODS:
    for cond, df_c in [('256×256', df_256_a), ('native ', df_nat_a)]:
        pc = per_class_iou(df_c, mkey)
        print(f'{METHOD_LABELS[mkey]:<28}  {cond:>7}', end='')
        for c in CLASS_NAMES:
            print(f'  {pc.get(c, float("nan")):>10.4f}', end='')
        print()

## 10. Factorial Decomposition: Native vs 256×256

In [ ]:
def factorial_decomp(df: pd.DataFrame, label: str) -> Dict[str, float]:
    sg = macro_iou(df, 'std_gradcam')
    fg = macro_iou(df, 'fpn_gradcam')
    sl = macro_iou(df, 'std_layercam')
    fl = macro_iou(df, 'fpn_layercam')
    return {
        'label'          : label,
        'SG'             : round(sg, 4),
        'FG'             : round(fg, 4),
        'SL'             : round(sl, 4),
        'FL'             : round(fl, 4),
        'layercam_effect': round(((sl-sg)+(fl-fg))/2, 4),
        'pyramid_effect' : round(((fg-sg)+(fl-sl))/2, 4),
        'interaction'    : round((fl-fg)-(sl-sg), 4),
    }


f256 = factorial_decomp(df_256_a, '256×256')
fnat = factorial_decomp(df_nat_a, 'Native')

df_factorial = pd.DataFrame([f256, fnat])
df_factorial.to_csv(OUT_DIR / 'factorial_decomp_resolution.csv', index=False)

print('Factorial decomposition (IoU@0.50):')
print(f'{"":<22}  {"256×256":>10}  {"Native":>10}  {"Δ":>10}')
print('-' * 58)
for key in ['SG','FG','SL','FL','layercam_effect','pyramid_effect','interaction']:
    label = {
        'SG': 'Std GradCAM (SG)',
        'FG': 'FPN-GradCAM (FG)',
        'SL': 'Std LayerCAM (SL)',
        'FL': 'FPN-LayerCAM (FL)',
        'layercam_effect': 'LayerCAM effect',
        'pyramid_effect' : 'Pyramid  effect  ← key',
        'interaction'    : 'Interaction term',
    }[key]
    v256 = f256[key]
    vnat = fnat[key]
    delta = vnat - v256
    flag = '  *** REVERSAL' if key=='pyramid_effect' and v256<0 and vnat>0 else \
           '  ** REDUCTION' if key=='pyramid_effect' and v256<0 and vnat<0 and vnat>v256 else ''
    print(f'{label:<22}  {v256:>+10.4f}  {vnat:>+10.4f}  {delta:>+10.4f}{flag}')

print(f'\nThe key test: does pyramid_effect flip from negative to positive at native res?')
pyr_256 = f256['pyramid_effect']
pyr_nat = fnat['pyramid_effect']
if pyr_256 < 0 and pyr_nat > 0:
    print(f'  YES — pyramid effect reversed: {pyr_256:+.4f} (256×256) → {pyr_nat:+.4f} (native)')
    print('  This CONFIRMS the resolution hypothesis: downsampling caused the reversal.')
elif pyr_256 < 0 and pyr_nat < 0 and pyr_nat > pyr_256:
    print(f'  PARTIAL — still negative but reduced: {pyr_256:+.4f} → {pyr_nat:+.4f}')
    print('  Consistent with hypothesis but reversal incomplete.')
elif pyr_256 < 0 and pyr_nat < 0 and pyr_nat <= pyr_256:
    print(f'  NO CHANGE / WORSE: {pyr_256:+.4f} → {pyr_nat:+.4f}')
    print('  Native resolution does NOT recover the FPN benefit.')
else:
    print(f'  256×256 effect was already positive ({pyr_256:+.4f}). Native: {pyr_nat:+.4f}.')

## 11. Scale-Factor Analysis

The resolution hypothesis predicts that the FPN improvement should be
largest on images where the native resolution is much larger than 256 px
(high scale factor = most spatial information destroyed by resizing).

In [ ]:
# Merge native and 256 results to compare per-image deltas
merge_cols_256 = ['global_idx'] + [
    f'{m}_iou_0.50_{c}' for m in METHODS for c in CLASS_NAMES
]
merge_cols_256 = [c for c in merge_cols_256 if c in df_256_a.columns]

# Rename 256 columns to avoid collision
df_256_renamed = df_256_a[merge_cols_256].rename(
    columns={c: c.replace('_iou_0.50_', '_256_iou_0.50_')
             for c in merge_cols_256 if '_iou_0.50_' in c}
)

df_merged = df_nat_a.merge(df_256_renamed, on='global_idx', how='inner')

# Compute per-image FPN delta (FPN-LayerCAM - Std GradCAM) for each condition
fpn_cols_nat = [f'fpn_layercam_iou_0.50_{c}' for c in CLASS_NAMES
                if f'fpn_layercam_iou_0.50_{c}' in df_merged.columns]
std_cols_nat = [f'std_gradcam_iou_0.50_{c}'  for c in CLASS_NAMES
                if f'std_gradcam_iou_0.50_{c}'  in df_merged.columns]

df_merged['fpn_macro_nat'] = df_merged[fpn_cols_nat].mean(axis=1)
df_merged['std_macro_nat'] = df_merged[std_cols_nat].mean(axis=1)
df_merged['fpn_gain_nat']  = df_merged['fpn_macro_nat'] - df_merged['std_macro_nat']

fpn_cols_256 = [f'fpn_layercam_256_iou_0.50_{c}' for c in CLASS_NAMES
                if f'fpn_layercam_256_iou_0.50_{c}' in df_merged.columns]
std_cols_256 = [f'std_gradcam_256_iou_0.50_{c}'  for c in CLASS_NAMES
                if f'std_gradcam_256_iou_0.50_{c}'  in df_merged.columns]

if fpn_cols_256 and std_cols_256:
    df_merged['fpn_macro_256'] = df_merged[fpn_cols_256].mean(axis=1)
    df_merged['std_macro_256'] = df_merged[std_cols_256].mean(axis=1)
    df_merged['fpn_gain_256']  = df_merged['fpn_macro_256'] - df_merged['std_macro_256']

df_merged.to_csv(OUT_DIR / 'per_image_comparison.csv', index=False)

# Spearman correlation: scale_factor vs FPN gain at native resolution
valid = df_merged[['scale_factor', 'fpn_gain_nat']].dropna()
if len(valid) >= 4:
    rho, p = stats.spearmanr(valid['scale_factor'], valid['fpn_gain_nat'])
    print(f'Spearman rho (scale_factor vs FPN gain at native res): {rho:+.4f}  p={p:.4f}')
    print('Positive rho = larger images benefit MORE from FPN at native res')
    print('Negative rho = larger images STILL do not benefit from FPN')

print(f'\nScale factor distribution:')
print(df_merged['scale_factor'].describe().round(2))

## 12. Figure 1 — IoU@0.50 Bar Chart: Native vs 256×256

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=False)
fig.suptitle(
    f'Resolution Ablation: Native vs 256×256 Input — MoNuSAC (n={len(common_gidx)})\n'
    'Solid = native resolution  |  Hatched = 256×256 (baseline)',
    fontsize=12, fontweight='bold',
)

for ax, (cond, df_c, hatch, alpha) in zip(axes, [
    ('Native',  df_nat_a, '',    0.92),
    ('256×256', df_256_a, '///', 0.65),
]):
    x = np.arange(NUM_CLASSES)
    w = 0.18
    offsets = np.linspace(-1.5*w, 1.5*w, 4)
    for mkey, off in zip(METHODS, offsets):
        pc = per_class_iou(df_c, mkey)
        vals = [pc.get(c, float('nan')) for c in CLASS_NAMES]
        bars = ax.bar(x + off, vals, w, label=METHOD_LABELS[mkey],
                      color=METHOD_COLORS[mkey], hatch=hatch,
                      edgecolor='white', linewidth=0.4, alpha=alpha)
        for b, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.002,
                        f'{v:.3f}', ha='center', va='bottom',
                        fontsize=6.5, rotation=90)
    ax.set_xticks(x); ax.set_xticklabels(CLASS_DISPLAY, rotation=15, ha='right', fontsize=9)
    ax.set_title(f'{cond} Resolution', fontsize=11, fontweight='bold')
    ax.set_ylabel('Mean IoU@0.50', fontsize=9)
    ax.set_ylim(0, 0.25)
    ax.legend(fontsize=7.5, loc='upper right', framealpha=0.6)
    ax.grid(axis='y', alpha=0.25, linestyle=':')
    # Annotate macro
    for mkey in METHODS:
        m = macro_iou(df_c, mkey)
        ax.text(0.01, 0.96 - METHODS.index(mkey)*0.055,
                f'{METHOD_LABELS[mkey][:12]} macro={m:.4f}',
                transform=ax.transAxes, fontsize=6.5, color=METHOD_COLORS[mkey],
                fontweight='bold')

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig1_iou_native_vs_256.{ext}',
                bbox_inches='tight', dpi=300)
plt.show()
print('Figure 1 saved.')

## 13. Figure 2 — Factorial Decomposition: Native vs 256×256

In [ ]:
effects   = ['layercam_effect', 'pyramid_effect', 'interaction']
eff_label = ['LayerCAM\nmain effect', 'Pyramid\nmain effect', 'Interaction\nterm']

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle(
    'Factorial Decomposition — Native vs 256×256\n'
    'Pyramid main effect is the key test of the resolution hypothesis',
    fontsize=12, fontweight='bold',
)
x = np.arange(len(effects))
w = 0.32
conds = [('256×256', f256, '#90CAF9', '///'), ('Native', fnat, '#A5D6A7', '')]
for i, (cond, fd, color, hatch) in enumerate(conds):
    vals = [fd[e] for e in effects]
    bars = ax.bar(x + (i-0.5)*w, vals, w, label=cond,
                  color=color, hatch=hatch, edgecolor='#555555',
                  linewidth=0.6, alpha=0.9)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2,
                b.get_height() + (0.001 if v>=0 else -0.004),
                f'{v:+.4f}', ha='center',
                va='bottom' if v>=0 else 'top', fontsize=9, fontweight='bold')

ax.axhline(0, color='black', linewidth=0.8, linestyle='-')
ax.set_xticks(x); ax.set_xticklabels(eff_label, fontsize=11)
ax.set_ylabel('IoU@0.50 effect size', fontsize=10)
ax.legend(fontsize=10, framealpha=0.7)
ax.grid(axis='y', alpha=0.25, linestyle=':')

# Annotate the pyramid bar pair
pyr_i = effects.index('pyramid_effect')
ax.annotate('Resolution\nhypothesis test',
            xy=(pyr_i, max(fnat['pyramid_effect'], 0) + 0.002),
            xytext=(pyr_i + 0.5, max(fnat['pyramid_effect'], f256['pyramid_effect']) + 0.012),
            fontsize=8, color='#333333',
            arrowprops=dict(arrowstyle='->', color='#333333', lw=1.2))

plt.tight_layout()
for ext in ['pdf', 'png']:
    plt.savefig(FIG_DIR / f'fig2_factorial_native_vs_256.{ext}',
                bbox_inches='tight', dpi=300)
plt.show()
print('Figure 2 saved.')

## 14. Figure 3 — Per-Image FPN Gain vs Scale Factor

In [ ]:
valid_merged = df_merged[['scale_factor', 'fpn_gain_nat']].dropna()

if len(valid_merged) >= 4:
    fig, ax = plt.subplots(figsize=(9, 5))
    fig.suptitle(
        'Per-Image FPN Gain (FPN-LayerCAM − Std GradCAM) vs Scale Factor\n'
        'at Native Resolution — MoNuSAC',
        fontsize=11, fontweight='bold',
    )
    ax.scatter(valid_merged['scale_factor'], valid_merged['fpn_gain_nat'],
               color='#A5D6A7', edgecolors='#333333', linewidths=0.5,
               s=60, zorder=3)

    # Regression line
    m_slope, m_int, rho, p, _ = stats.linregress(
        valid_merged['scale_factor'], valid_merged['fpn_gain_nat']
    )
    xs = np.linspace(valid_merged['scale_factor'].min(),
                     valid_merged['scale_factor'].max(), 100)
    ax.plot(xs, m_slope * xs + m_int, 'r--', linewidth=1.5,
            label=f'OLS  ρ={rho:+.3f}  p={p:.3f}')

    ax.axhline(0, color='black', linewidth=0.8, linestyle='-', alpha=0.5)
    ax.set_xlabel('Scale factor (native_max / 256)', fontsize=10)
    ax.set_ylabel('FPN gain (FPN-LayerCAM − SG) @ native res', fontsize=10)
    ax.legend(fontsize=9); ax.grid(alpha=0.25, linestyle=':')

    plt.tight_layout()
    for ext in ['pdf', 'png']:
        plt.savefig(FIG_DIR / f'fig3_fpn_gain_vs_scale.{ext}',
                    bbox_inches='tight', dpi=300)
    plt.show()
    print(f'Figure 3 saved.  rho={rho:+.4f}  p={p:.4f}')
else:
    print(f'Only {len(valid_merged)} valid rows — scatter plot skipped.')

## 15. Final Summary

In [ ]:
print('=' * 70)
print('RESOLUTION ABLATION SUMMARY — MoNuSAC')
print('=' * 70)

print(f'\n(1) Macro IoU@0.50  ({len(common_gidx)} test images):')
print(f'{"Method":<28}  {"256×256":>8}  {"Native":>8}  {"Δ":>8}')
print('-' * 58)
for mkey in METHODS:
    v256 = macro_iou(df_256_a, mkey)
    vnat = macro_iou(df_nat_a, mkey)
    print(f'{METHOD_LABELS[mkey]:<28}  {v256:>8.4f}  {vnat:>8.4f}  {vnat-v256:>+8.4f}')

print(f'\n(2) Factorial decomposition comparison:')
print(f'{"Effect":<22}  {"256×256":>10}  {"Native":>10}')
print('-' * 48)
for e, lbl in zip(effects, ['LayerCAM effect','Pyramid  effect','Interaction']):
    print(f'{lbl:<22}  {f256[e]:>+10.4f}  {fnat[e]:>+10.4f}')

print(f'\n(3) Resolution hypothesis verdict:')
if fnat['pyramid_effect'] > f256['pyramid_effect']:
    print(f'  Pyramid effect improved at native resolution: '
          f'{f256["pyramid_effect"]:+.4f} → {fnat["pyramid_effect"]:+.4f}')
    if f256['pyramid_effect'] < 0 <= fnat['pyramid_effect']:
        print('  FULL REVERSAL confirmed — hypothesis strongly supported.')
    else:
        print('  Partial improvement — hypothesis directionally supported.')
else:
    print(f'  No improvement in pyramid effect: '
          f'{f256["pyramid_effect"]:+.4f} → {fnat["pyramid_effect"]:+.4f}')
    print('  Additional factors beyond resolution are affecting FPN performance.')

print(f'\n(4) Output files:')
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file():
        print(f'  {str(f.relative_to(OUT_DIR)):<50}  {f.stat().st_size/1024:>6.1f} KB')